In [1]:

import numpy as np
from PIL import Image
from tqdm import tqdm
from cyvlfeat.sift.dsift import dsift
from cyvlfeat.kmeans import kmeans
from scipy.spatial.distance import cdist
import statistics

import os

In [2]:
import os
import argparse
import pickle
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from sklearn.metrics import confusion_matrix

from utils import get_tiny_images
from utils import build_vocabulary, get_bags_of_sifts
from utils import nearest_neighbor_classify
CAT = ['Kitchen', 'Store', 'Bedroom', 'LivingRoom', 'Office',
       'Industrial', 'Suburb', 'InsideCity', 'TallBuilding', 'Street',
       'Highway', 'OpenCountry', 'Coast', 'Mountain', 'Forest']

ABBR_CAT = ['Kit', 'Sto', 'Bed', 'Liv', 'Off',
            'Ind', 'Sub', 'Cty', 'Bld', 'St',
            'HW', 'OC', 'Cst', 'Mnt', 'For']

CAT2ID = {v: k for k, v in enumerate(CAT)}

NUM_PER_CAT = 100 # number of training or testing examples per category
assert NUM_PER_CAT <= 100


def get_img_paths_and_labels(data_path):

    train_img_paths, test_img_paths = [], []
    train_labels, test_labels = [], []

    assert os.path.exists(data_path), f'Invalid data path: {data_path}'

    for cat in CAT:
        cat_train_img_paths = glob(os.path.join(data_path, 'train', cat, '*.jpg'))
        cat_test_img_paths = glob(os.path.join(data_path, 'test', cat, '*.jpg'))
        for i in range(NUM_PER_CAT):
            train_img_paths.append(cat_train_img_paths[i])
            test_img_paths.append(cat_test_img_paths[i])
            train_labels.append(cat)
            test_labels.append(cat)

    # (1500), (1500), (1500), (1500)
    return train_img_paths, test_img_paths, train_labels, test_labels


In [3]:
train_img_paths, test_img_paths, train_labels, test_labels = get_img_paths_and_labels('/mnt/20F408ADF408876E/114_2/computer_vision/hw2_data/p1_data/')
sample_path = train_img_paths[0]
print(sample_path)

/mnt/20F408ADF408876E/114_2/computer_vision/hw2_data/p1_data/train/Kitchen/image_0001.jpg


In [4]:
vocab = build_vocabulary(train_img_paths, 100, debug=True)

100%|██████████| 1500/1500 [00:06<00:00, 218.37it/s]


In [5]:
vocab_pool = []

stepsize = 10
for img_path in tqdm(train_img_paths):
    img = Image.open(img_path)
    img = np.array(img)
    _, descriptions = dsift(img, step = [stepsize, stepsize], fast = True)

    for desp in descriptions:
        vocab_pool.append( desp.astype(np.float32) )

100%|██████████| 1500/1500 [00:05<00:00, 268.68it/s]


In [ ]:
vocab_pool = np.array(vocab_pool)
vocab = kmeans(vocab_pool, num_centers=100, verbose = True)

In [ ]:
print('Loading all data paths and labels...')
train_img_paths, test_img_paths, train_labels, test_labels = get_img_paths_and_labels('/mnt/20F408ADF408876E/114_2/computer_vision/hw2_data/p1_data/')

    # vocab and features is saved to disk to avoid recomputing the vocabulary every time
    if os.path.isfile('vocab.pkl') is False:
        print('No existing visual word vocabulary found. Computing one from training images')
        vocab_size = 400  # vocab_size is up to you, larger values will work better (to a point) but be slower to compute
        vocab = build_vocabulary(train_img_paths, vocab_size)
        with open('vocab.pkl', 'wb') as f:
            pickle.dump(vocab, f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open('vocab.pkl', 'rb') as f:
            vocab = pickle.load(f)
    # train_image_feats
    if os.path.isfile('train_image_feats.pkl') is False:
        train_img_feats = get_bags_of_sifts(train_img_paths, vocab)
        with open('train_image_feats.pkl', 'wb') as f:
            pickle.dump(train_img_feats, f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open('train_image_feats.pkl', 'rb') as f:
            train_img_feats = pickle.load(f)
    # test_image_feats
    if os.path.isfile('test_image_feats.pkl') is False:
        test_img_feats  = get_bags_of_sifts(test_img_paths, vocab)
        with open('test_image_feats.pkl', 'wb') as f:
            pickle.dump(test_img_feats, f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open('test_image_feats.pkl', 'rb') as f:
            test_img_feats = pickle.load(f)

else:
    raise NameError('Unknown feature type')

##########################################
###### Step 2: Classify Test Images ######
##########################################
##### TODO: nearest_neighbor_classify() in utils.py #####
print(f'Classifier: {'nearest_neighbor'}')
if 'nearest_neighbor' == 'nearest_neighbor':
    pred_cats = nearest_neighbor_classify(train_img_feats,
                                            train_labels,
                                            test_img_feats)

elif 'nearest_neighbor' == 'random_classifier':
    pred_cats = test_labels[:]
    np.random.shuffle(pred_cats)

else:
    raise NameError('Unknown classifier type')

# Compute Accuracy (DO NOT MODIFY)
accuracy = float(len([x for x in zip(test_labels, pred_cats) if x[0] == x[1]])) / float(len(test_labels))
print(f'Accuracy = {accuracy}\n')
# Plot Confusion Matrix (You don't need to write this code)
test_labels_ids = [CAT2ID[x] for x in test_labels] # (1500)
pred_cats_ids   = [CAT2ID[x] for x in pred_cats  ] # (1500)

if pred_cats_ids:
    plot_confusion_mtx(test_labels_ids, pred_cats_ids, 'bag_of_sift')
